In [1]:
#!/usr/bin/env python3
# =========================================================
# FACTORIZATION MACHINE FOR YELP DATASET
# CuPy Implementation - KAGGLE VERSION
# Fixes:
#   [1] Dropout mask consistency (forward == backward)
#   [2] Vectorized scatter-add (no Python for-loops on GPU)
#   [3] Kaggle paths
# =========================================================

import json
import cupy as cp
import numpy as np
import math
import random
from datetime import datetime
import time


# =========================================================
# ID MAPPER
# =========================================================
class IDMapper:
    """Map string IDs to integer indices"""
    def __init__(self):
        self.map = {}

    def get(self, key):
        if key not in self.map:
            self.map[key] = len(self.map)
        return self.map[key]

    def size(self):
        return len(self.map)


# =========================================================
# LOAD USER CONTEXT
# =========================================================
def load_user_context(path):
    """Load user features from JSON"""
    ctx = {}
    print(f"Loading user context from {path}...")

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            u = json.loads(line)
            ctx[u["user_id"]] = [
                max(0, u.get("review_count", 0) or 0),
                max(0, u.get("average_stars", 0) or 0),
                max(0, u.get("fans", 0) or 0),
            ]

    print(f"  ✓ Loaded {len(ctx)} users")
    return ctx


# =========================================================
# LOAD BUSINESS CONTEXT
# =========================================================
def load_business_context(path):
    """Load business features from JSON"""
    ctx = {}
    print(f"Loading business context from {path}...")

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            b = json.loads(line)
            ctx[b["business_id"]] = [
                max(0, b.get("stars", 0) or 0),
                max(0, b.get("review_count", 0) or 0),
                max(0, b.get("is_open", 0) or 0),
            ]

    print(f"  ✓ Loaded {len(ctx)} businesses")
    return ctx


# =========================================================
# LOAD CHECKIN CONTEXT
# =========================================================
def load_checkin_context(path):
    """Load checkin features from JSON"""
    ctx = {}
    print(f"Loading checkin context from {path}...")

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            c = json.loads(line)
            bid = c["business_id"]
            dates = c.get("date", "")
            checkin_count = len(dates.split(", ")) if dates else 0
            ctx[bid] = [checkin_count]

    print(f"  ✓ Loaded {len(ctx)} businesses with checkin data")
    return ctx


# =========================================================
# LOAD TIP CONTEXT
# =========================================================
def load_tip_context(path):
    """Load tip features from JSON"""
    user_tip = {}
    biz_tip = {}

    print(f"Loading tip context from {path}...")

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            t = json.loads(line)
            uid = t["user_id"]
            bid = t["business_id"]

            length = len(t.get("text", ""))
            comp = max(0, t.get("compliment_count", 0) or 0)

            if uid not in user_tip:
                user_tip[uid] = [0, 0, 0]
            user_tip[uid][0] += 1
            user_tip[uid][1] += length
            user_tip[uid][2] += comp

            if bid not in biz_tip:
                biz_tip[bid] = [0, 0, 0]
            biz_tip[bid][0] += 1
            biz_tip[bid][1] += length
            biz_tip[bid][2] += comp

    print(f"  ✓ Loaded tips for {len(user_tip)} users and {len(biz_tip)} businesses")
    return user_tip, biz_tip


# =========================================================
# DATA LOADER
# =========================================================
def load_reviews(review_path, user_ctx, biz_ctx, checkin_ctx, user_tip, biz_tip,
                 user_map, biz_map, max_reviews=None):
    """
    Load and preprocess reviews into memory
    Returns: (user_indices, biz_indices, numerical_features, ratings)
    """
    print(f"\nLoading reviews from {review_path}...")
    if max_reviews:
        print(f"  ⚠️  Limited to first {max_reviews:,} reviews")

    user_indices = []
    biz_indices = []
    num_features = []
    ratings = []

    count = 0

    with open(review_path, "r", encoding="utf-8") as f:
        for line in f:
            if max_reviews and count >= max_reviews:
                break

            r = json.loads(line)

            uid = r["user_id"]
            bid = r["business_id"]

            uidx = user_map.get(uid)
            bidx = biz_map.get(bid)

            # USER CONTEXT
            uctx = user_ctx.get(uid, [0, 0, 0])

            # BUSINESS CONTEXT
            bctx = biz_ctx.get(bid, [0, 0, 0])

            # TIP FEATURES
            utip = user_tip.get(uid, [0, 0, 0])
            btip = biz_tip.get(bid, [0, 0, 0])

            utip_feat = [
                utip[0],
                utip[1] / max(utip[0], 1),
                utip[2] / max(utip[0], 1),
            ]

            btip_feat = [
                btip[0],
                btip[1] / max(btip[0], 1),
                btip[2] / max(btip[0], 1),
            ]

            # CHECKIN FEATURES
            checkin = checkin_ctx.get(bid, [0])

            # REVIEW FEATURES
            useful = max(0, int(r.get("useful", 0) or 0))
            funny = max(0, int(r.get("funny", 0) or 0))
            cool = max(0, int(r.get("cool", 0) or 0))
            text_len = len(r.get("text", ""))

            # TEMPORAL FEATURES
            dt = datetime.strptime(r["date"], "%Y-%m-%d %H:%M:%S")
            year_norm = (dt.year - 2015) / 10.0
            month_norm = dt.month / 12.0
            weekday_norm = dt.weekday() / 6.0

            # NORMALIZE ALL FEATURES
            text_len_norm = math.log1p(text_len) / 10.0
            useful_norm = math.log1p(useful)
            funny_norm = math.log1p(funny)
            cool_norm = math.log1p(cool)

            uctx_norm = [
                math.log1p(uctx[0]) / 10.0,
                uctx[1] / 5.0,
                math.log1p(uctx[2]) / 10.0,
            ]

            bctx_norm = [
                bctx[0] / 5.0,
                math.log1p(bctx[1]) / 10.0,
                float(bctx[2]),
            ]

            utip_feat_norm = [
                math.log1p(utip_feat[0]) / 10.0,
                math.log1p(utip_feat[1]) / 10.0,
                math.log1p(utip_feat[2]) / 10.0,
            ]

            btip_feat_norm = [
                math.log1p(btip_feat[0]) / 10.0,
                math.log1p(btip_feat[1]) / 10.0,
                math.log1p(btip_feat[2]) / 10.0,
            ]

            checkin_norm = math.log1p(checkin[0]) / 10.0

            # COMBINE ALL NUMERICAL FEATURES
            num_feat = np.array(
                uctx_norm + bctx_norm + utip_feat_norm + btip_feat_norm +
                [checkin_norm] +
                [useful_norm, funny_norm, cool_norm, text_len_norm] +
                [year_norm, month_norm, weekday_norm],
                dtype=np.float32
            )

            rating = float(r["stars"])

            user_indices.append(uidx)
            biz_indices.append(bidx)
            num_features.append(num_feat)
            ratings.append(rating)

            count += 1

            if count % 100000 == 0:
                print(f"  Loaded {count:,} reviews...")

    print(f"  ✓ Total: {count:,} reviews loaded")
    print(f"  ✓ Unique users: {user_map.size():,}")
    print(f"  ✓ Unique businesses: {biz_map.size():,}")
    print(f"  ✓ Numerical features per sample: {len(num_features[0])}")

    return (
        np.array(user_indices, dtype=np.int32),
        np.array(biz_indices, dtype=np.int32),
        np.array(num_features, dtype=np.float32),
        np.array(ratings, dtype=np.float32)
    )


# =========================================================
# FACTORIZATION MACHINE
# =========================================================
class FactorizationMachine:
    """
    FM với CuPy - KAGGLE VERSION
    Fixes:
      - Dropout mask nhất quán giữa forward và backward
      - Vectorized scatter-add (cp.add.at), không dùng Python for-loop
      - Gradient clipping đúng chiều
    """

    def __init__(self, n_users, n_businesses, num_dim, k=8,
                 learning_rate=0.001, reg=0.01, dropout_rate=0.4, grad_clip=5.0):
        self.n_users = n_users
        self.n_businesses = n_businesses
        self.num_dim = num_dim
        self.k = k
        self.lr = learning_rate
        self.reg = reg
        self.dropout_rate = dropout_rate
        self.grad_clip = grad_clip
        self.training = True

        print(f"\n🔧 Initializing FM model on GPU:")
        print(f"  Users: {n_users:,}, Businesses: {n_businesses:,}")
        print(f"  Numerical features: {num_dim}")
        print(f"  Embedding dimension: {k}")
        print(f"  Regularization: {reg}")
        print(f"  Dropout rate: {dropout_rate}")
        print(f"  Gradient clipping: {grad_clip}")

        # Global bias
        self.bias = cp.float32(0.0)

        # Linear weights
        self.user_lin = cp.zeros(n_users, dtype=cp.float32)
        self.biz_lin  = cp.zeros(n_businesses, dtype=cp.float32)
        self.num_lin  = cp.zeros(num_dim, dtype=cp.float32)

        # Embeddings
        scale = float(1.0 / math.sqrt(k))
        self.user_emb = cp.random.randn(n_users, k).astype(cp.float32) * scale
        self.biz_emb  = cp.random.randn(n_businesses, k).astype(cp.float32) * scale
        self.num_emb  = cp.random.randn(num_dim, k).astype(cp.float32) * scale

        print("  ✓ Model initialized on GPU")

    # ----------------------------------------------------------
    # Clip by global norm (works for scalars and arrays)
    # ----------------------------------------------------------
    def _clip(self, grad):
        norm = cp.linalg.norm(grad)
        if norm > self.grad_clip:
            grad = grad * (self.grad_clip / (norm + 1e-8))
        return grad

    # ----------------------------------------------------------
    # Predict only (no dropout for eval)
    # ----------------------------------------------------------
    def predict_batch(self, user_indices, biz_indices, num_features):
        batch_size = len(user_indices)

        linear = cp.full(batch_size, self.bias, dtype=cp.float32)
        linear += self.user_lin[user_indices]
        linear += self.biz_lin[biz_indices]
        linear += cp.sum(self.num_lin * num_features, axis=1)

        u_emb = self.user_emb[user_indices]
        b_emb = self.biz_emb[biz_indices]
        n_emb = cp.dot(num_features, self.num_emb)

        stacked  = cp.stack([u_emb, b_emb, n_emb], axis=1)   # (B, 3, k)
        sum_emb  = cp.sum(stacked, axis=1)                    # (B, k)
        interaction = 0.5 * cp.sum(sum_emb ** 2 - cp.sum(stacked ** 2, axis=1), axis=1)

        return linear + interaction

    # ----------------------------------------------------------
    # FIX 1: Forward + backward trong 1 hàm, dùng chung dropout mask
    # FIX 2: Vectorized scatter-add, không for-loop
    # ----------------------------------------------------------
    def sgd_step_batch(self, user_indices, biz_indices, num_features, ratings):
        batch_size = len(user_indices)

        # ===== FORWARD PASS =====
        linear = cp.full(batch_size, self.bias, dtype=cp.float32)
        linear += self.user_lin[user_indices]
        linear += self.biz_lin[biz_indices]
        linear += cp.sum(self.num_lin * num_features, axis=1)

        u_emb = self.user_emb[user_indices]          # (B, k)
        b_emb = self.biz_emb[biz_indices]            # (B, k)
        n_emb = cp.dot(num_features, self.num_emb)  # (B, k)

        # FIX 1: Tạo mask 1 lần, dùng lại cho backward
        if self.training and self.dropout_rate > 0:
            inv_keep = cp.float32(1.0 / (1.0 - self.dropout_rate))
            u_mask = (cp.random.rand(batch_size, self.k).astype(cp.float32) > self.dropout_rate) * inv_keep
            b_mask = (cp.random.rand(batch_size, self.k).astype(cp.float32) > self.dropout_rate) * inv_keep
            n_mask = (cp.random.rand(batch_size, self.k).astype(cp.float32) > self.dropout_rate) * inv_keep
            u_emb_d = u_emb * u_mask
            b_emb_d = b_emb * b_mask
            n_emb_d = n_emb * n_mask
        else:
            u_emb_d, b_emb_d, n_emb_d = u_emb, b_emb, n_emb
            u_mask = b_mask = n_mask = None

        stacked    = cp.stack([u_emb_d, b_emb_d, n_emb_d], axis=1)  # (B, 3, k)
        sum_emb    = cp.sum(stacked, axis=1)                          # (B, k)
        interaction = 0.5 * cp.sum(sum_emb ** 2 - cp.sum(stacked ** 2, axis=1), axis=1)

        predictions = linear + interaction
        errors      = predictions - ratings                           # (B,)
        loss        = cp.mean(errors ** 2)

        # Reg loss (chỉ dùng cho log, không ảnh hưởng gradient flow)
        reg_loss = self.reg * (
            cp.sum(self.user_lin[user_indices] ** 2) +
            cp.sum(self.biz_lin[biz_indices]   ** 2) +
            cp.sum(self.num_lin                ** 2) +
            cp.sum(self.user_emb[user_indices] ** 2) +
            cp.sum(self.biz_emb[biz_indices]   ** 2) +
            cp.sum(self.num_emb                ** 2)
        ) / batch_size
        total_loss = loss + reg_loss

        # ===== BACKWARD PASS =====
        e = errors / batch_size   # scaled error (B,)

        # --- Bias ---
        self.bias -= self.lr * cp.mean(errors)

        # --- Linear weights (FIX 2: cp.add.at thay for-loop) ---
        # user
        g_ul = cp.zeros(self.n_users, dtype=cp.float32)
        cp.add.at(g_ul, user_indices, e)
        g_ul += self.reg * self.user_lin
        self.user_lin -= self.lr * self._clip(g_ul)

        # business
        g_bl = cp.zeros(self.n_businesses, dtype=cp.float32)
        cp.add.at(g_bl, biz_indices, e)
        g_bl += self.reg * self.biz_lin
        self.biz_lin -= self.lr * self._clip(g_bl)

        # numerical
        g_nl = cp.dot(num_features.T, e) + self.reg * self.num_lin   # (num_dim,)
        self.num_lin -= self.lr * self._clip(g_nl)

        # --- Embedding gradients ---
        # dL/d(emb_i) = error * (sum_emb - emb_i_dropped)
        e_col = e[:, None]   # (B, 1) for broadcasting

        # user emb raw gradient (B, k), dùng u_emb_d đã apply mask
        g_u = e_col * (sum_emb - u_emb_d)
        if u_mask is not None:
            g_u = g_u * u_mask   # chain rule qua dropout

        g_u_acc = cp.zeros((self.n_users, self.k), dtype=cp.float32)
        cp.add.at(g_u_acc, user_indices, g_u)
        g_u_acc += self.reg * self.user_emb
        self.user_emb -= self.lr * self._clip(g_u_acc)

        # biz emb
        g_b = e_col * (sum_emb - b_emb_d)
        if b_mask is not None:
            g_b = g_b * b_mask

        g_b_acc = cp.zeros((self.n_businesses, self.k), dtype=cp.float32)
        cp.add.at(g_b_acc, biz_indices, g_b)
        g_b_acc += self.reg * self.biz_emb
        self.biz_emb -= self.lr * self._clip(g_b_acc)

        # num emb
        g_n = e_col * (sum_emb - n_emb_d)        # (B, k)
        if n_mask is not None:
            g_n = g_n * n_mask
        g_n_acc = cp.dot(num_features.T, g_n) + self.reg * self.num_emb  # (num_dim, k)
        self.num_emb -= self.lr * self._clip(g_n_acc)

        return float(total_loss)


# =========================================================
# EVALUATION METRICS
# =========================================================
def compute_metrics(predictions, targets):
    predictions = cp.asnumpy(predictions)
    targets     = cp.asnumpy(targets)

    mse  = np.mean((predictions - targets) ** 2)
    rmse = np.sqrt(mse)
    mae  = np.mean(np.abs(predictions - targets))

    ss_res = np.sum((targets - predictions) ** 2)
    ss_tot = np.sum((targets - np.mean(targets)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0

    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}


def print_metrics(metrics, prefix=""):
    print(f"{prefix}MSE: {metrics['MSE']:.4f} | "
          f"RMSE: {metrics['RMSE']:.4f} | "
          f"MAE: {metrics['MAE']:.4f} | "
          f"R2: {metrics['R2']:.4f}")


# =========================================================
# LEARNING RATE SCHEDULER
# =========================================================
def get_lr_decay(epoch, base_lr, decay_rate=0.99):
    return base_lr * (decay_rate ** epoch)

def predict_in_chunks(model, user_idx, biz_idx, num_feat, chunk_size=50000):
    preds = []
    for i in range(0, len(user_idx), chunk_size):
        s, e = i, min(i + chunk_size, len(user_idx))
        p = model.predict_batch(user_idx[s:e], biz_idx[s:e], num_feat[s:e])
        preds.append(p)
    return cp.concatenate(preds)
# =========================================================
# TRAINING
# =========================================================
def train_fm(user_indices, biz_indices, num_features, ratings,
             n_users, n_businesses, num_dim,
             epochs=50, batch_size=8192, val_split=0.1,
             base_lr=0.005, k=8, reg=0.01, dropout_rate=0.4):

    print("\n" + "="*70)
    print("🚀 TRAINING FACTORIZATION MACHINE (Kaggle Version)")
    print("="*70)

    # Split data
    n_samples = len(user_indices)
    indices   = np.arange(n_samples)
    np.random.shuffle(indices)

    val_size  = int(n_samples * val_split)
    val_idx   = indices[:val_size]
    train_idx = indices[val_size:]

    print(f"\nDataset split:")
    print(f"  Train: {len(train_idx):,} samples")
    print(f"  Val:   {len(val_idx):,}   samples")

    # Move to GPU
    print("\nMoving data to GPU...")
    user_indices_gpu  = cp.array(user_indices)
    biz_indices_gpu   = cp.array(biz_indices)
    num_features_gpu  = cp.array(num_features)
    ratings_gpu       = cp.array(ratings)
    print("  ✓ Data on GPU")

    # Baseline
    baseline_mean = float(cp.mean(ratings_gpu[train_idx]))
    baseline_rmse = float(cp.sqrt(cp.mean((ratings_gpu[val_idx] - baseline_mean) ** 2)))
    print(f"\n📊 Baseline (predict mean={baseline_mean:.2f}): RMSE = {baseline_rmse:.4f}")

    # Create model
    model = FactorizationMachine(
        n_users=n_users,
        n_businesses=n_businesses,
        num_dim=num_dim,
        k=k,
        learning_rate=base_lr,
        reg=reg,
        dropout_rate=dropout_rate
    )

    history = {'train_loss': [], 'val_rmse': [], 'lr': []}

    best_val_rmse    = float('inf')
    best_model_state = None
    patience         = 15
    patience_counter = 0

    print(f"\nTraining configuration:")
    print(f"  Epochs: {epochs} | Batch: {batch_size} | LR: {base_lr}")
    print(f"  Reg: {reg} | Dropout: {dropout_rate} | Patience: {patience}")
    print("\n" + "-"*70)

    for epoch in range(epochs):
        epoch_start = time.time()

        current_lr  = get_lr_decay(epoch, base_lr, decay_rate=0.99)
        model.lr    = current_lr
        history['lr'].append(current_lr)

        model.training = True
        np.random.shuffle(train_idx)

        total_loss = 0.0
        n_batches  = (len(train_idx) + batch_size - 1) // batch_size

        for batch_num in range(n_batches):
            s = batch_num * batch_size
            e = min(s + batch_size, len(train_idx))
            idx = train_idx[s:e]

            batch_loss = model.sgd_step_batch(
                user_indices_gpu[idx],
                biz_indices_gpu[idx],
                num_features_gpu[idx],
                ratings_gpu[idx]
            )
            total_loss += batch_loss * (e - s)

            if math.isnan(batch_loss) or batch_loss > 1000:
                print(f"\n⚠️  WARNING: Loss exploded! ({batch_loss:.2f}) — stopping.")
                return model, history

        avg_loss = total_loss / len(train_idx)
        history['train_loss'].append(avg_loss)

        # Validation
        model.training = False
        val_preds   = model.predict_batch(
            user_indices_gpu[val_idx],
            biz_indices_gpu[val_idx],
            num_features_gpu[val_idx]
        )
        val_metrics = compute_metrics(val_preds, ratings_gpu[val_idx])
        history['val_rmse'].append(val_metrics['RMSE'])

        epoch_time     = time.time() - epoch_start
        beating_base   = "✓" if val_metrics['RMSE'] < baseline_rmse else "✗"

        print(f"Epoch {epoch+1:3d}/{epochs} | "
              f"Loss: {avg_loss:7.4f} | "
              f"Val RMSE: {val_metrics['RMSE']:.4f} {beating_base} | "
              f"MAE: {val_metrics['MAE']:.4f} | "
              f"R2: {val_metrics['R2']:7.4f} | "
              f"LR: {current_lr:.6f} | "
              f"{epoch_time:.1f}s")

        # Early stopping + save best
        if val_metrics['RMSE'] < best_val_rmse:
            best_val_rmse    = val_metrics['RMSE']
            patience_counter = 0
            best_model_state = {
                'bias':     cp.asnumpy(model.bias),
                'user_lin': cp.asnumpy(model.user_lin),
                'biz_lin':  cp.asnumpy(model.biz_lin),
                'num_lin':  cp.asnumpy(model.num_lin),
                'user_emb': cp.asnumpy(model.user_emb),
                'biz_emb':  cp.asnumpy(model.biz_emb),
                'num_emb':  cp.asnumpy(model.num_emb),
            }
            print(f"  🎯 New best! RMSE: {best_val_rmse:.4f} "
                  f"(Δ vs baseline: {baseline_rmse - best_val_rmse:+.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n⏹  Early stopping at epoch {epoch+1} "
                      f"(best val RMSE: {best_val_rmse:.4f})")
                if best_model_state:
                    model.bias     = cp.array(best_model_state['bias'])
                    model.user_lin = cp.array(best_model_state['user_lin'])
                    model.biz_lin  = cp.array(best_model_state['biz_lin'])
                    model.num_lin  = cp.array(best_model_state['num_lin'])
                    model.user_emb = cp.array(best_model_state['user_emb'])
                    model.biz_emb  = cp.array(best_model_state['biz_emb'])
                    model.num_emb  = cp.array(best_model_state['num_emb'])
                    print("   ✓ Restored best model")
                break

    # Final evaluation
    print("\n" + "="*70)
    print("FINAL EVALUATION")
    print("="*70)

    model.training = False

    train_preds = predict_in_chunks(model,
        user_indices_gpu[train_idx],
        biz_indices_gpu[train_idx],
        num_features_gpu[train_idx])
    train_metrics = compute_metrics(train_preds, ratings_gpu[train_idx])

    val_preds = predict_in_chunks(model,
        user_indices_gpu[val_idx],
        biz_indices_gpu[val_idx],
        num_features_gpu[val_idx])
    val_metrics = compute_metrics(val_preds, ratings_gpu[val_idx])

    print(f"\n📊 Baseline RMSE: {baseline_rmse:.4f}")
    print("\n📈 Training Set:");  print_metrics(train_metrics, "  ")
    print("\n📈 Validation Set:"); print_metrics(val_metrics, "  ")

    improvement = baseline_rmse - val_metrics['RMSE']
    print(f"\n{'✓' if improvement > 0 else '✗'} Improvement over baseline: {improvement:+.4f}")

    overfit = train_metrics['RMSE'] - val_metrics['RMSE']
    print(f"📉 Overfitting gap (Train-Val RMSE): {overfit:.4f}")
    if abs(overfit) < 0.3:
        print("   ✓ Good generalization!")
    elif abs(overfit) < 0.5:
        print("   ⚠️  Moderate overfitting")
    else:
        print("   ✗ High overfitting")

    return model, history


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":

    # ✅ Kaggle path (dataset name: yelp-dataset)
    BASE = "/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset"

    print("="*70)
    print("🚀 FM - KAGGLE VERSION (1M reviews)")
    print("="*70 + "\n")

    # Load contexts
    user_ctx              = load_user_context(f"{BASE}/yelp_academic_dataset_user.json")
    biz_ctx               = load_business_context(f"{BASE}/yelp_academic_dataset_business.json")
    checkin_ctx           = load_checkin_context(f"{BASE}/yelp_academic_dataset_checkin.json")
    user_tip, biz_tip     = load_tip_context(f"{BASE}/yelp_academic_dataset_tip.json")

    user_map = IDMapper()
    biz_map  = IDMapper()

    # Load 1M reviews
    user_indices, biz_indices, num_features, ratings = load_reviews(
        review_path=f"{BASE}/yelp_academic_dataset_review.json",
        user_ctx=user_ctx,
        biz_ctx=biz_ctx,
        checkin_ctx=checkin_ctx,
        user_tip=user_tip,
        biz_tip=biz_tip,
        user_map=user_map,
        biz_map=biz_map,
        max_reviews=1_000_000
    )

    # Train
    model, history = train_fm(
        user_indices=user_indices,
        biz_indices=biz_indices,
        num_features=num_features,
        ratings=ratings,
        n_users=user_map.size(),
        n_businesses=biz_map.size(),
        num_dim=num_features.shape[1],
        epochs=100,
        batch_size=8192,
        val_split=0.1,
        base_lr=0.01,
        k=64,
        reg=0.001,
        dropout_rate=0.3
    )

    print("\n✓ Training completed!")

    # ✅ Save to /kaggle/working/ (Kaggle output directory)
    import os
    os.makedirs('/kaggle/working/models', exist_ok=True)

    np.savez('/kaggle/working/models/fm_1M.npz',
        bias=cp.asnumpy(model.bias),
        user_lin=cp.asnumpy(model.user_lin),
        biz_lin=cp.asnumpy(model.biz_lin),
        num_lin=cp.asnumpy(model.num_lin),
        user_emb=cp.asnumpy(model.user_emb),
        biz_emb=cp.asnumpy(model.biz_emb),
        num_emb=cp.asnumpy(model.num_emb)
    )
    print("  ✓ Model saved to /kaggle/working/models/fm_1M.npz")

# Convert float32 -> float trước khi dump JSON
    history_serializable = {
        k: [float(v) for v in vals]
        for k, vals in history.items()
    }
    with open('/kaggle/working/models/training_history.json', 'w') as f:
        json.dump(history_serializable, f, indent=2)

🚀 FM - KAGGLE VERSION (1M reviews)

Loading user context from /kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_user.json...
  ✓ Loaded 1987897 users
Loading business context from /kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_business.json...
  ✓ Loaded 150346 businesses
Loading checkin context from /kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_checkin.json...
  ✓ Loaded 131930 businesses with checkin data
Loading tip context from /kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_tip.json...
  ✓ Loaded tips for 301758 users and 106193 businesses

Loading reviews from /kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_review.json...
  ⚠️  Limited to first 1,000,000 reviews
  Loaded 100,000 reviews...
  Loaded 200,000 reviews...
  Loaded 300,000 reviews...
  Loaded 400,000 reviews...
  Loaded 500,000 reviews.